# Modelo MIP para el calendario de la Premier League

Este notebook explica la formulación y permite ejecutar una temporada seleccionada. La implementación reutilizable está en [modelo_calendario.py](../modelo_calendario.py); el CSV histórico aporta la lista de clubes y no fija jornadas ni obliga a conservar los partidos históricos.

## Conjuntos y decisiones

Sean $E$ el conjunto de equipos y $J=\{1,\ldots,2(|E|-1)\}$ el conjunto de jornadas. $P$ contiene pares de clásicos, $S$ pares que comparten estadio, $E^{UEFA}$ los equipos con compromisos europeos y $J^c$ las jornadas críticas. La variable binaria $x_{ijt}$ vale 1 si $i$ recibe a $j$ en $t$; $y_{it}$ vale 1 si $i$ juega de local. Las variables $b^H_{it}$ y $b^A_{it}$ identifican breaks consecutivos de local y de visita. La variable $v_{it}$ marca que un equipo europeo juega de visita en una jornada crítica; es un proxy operativo, no un cálculo de distancia.

## Función objetivo

$$\min\quad w_b\sum_{i\in E}\sum_{t=2}^{|J|}(b^H_{it}+b^A_{it})+w_v\sum_{i\in E^{UEFA}}\sum_{t\in J^c}v_{it}$$

El primer término minimiza los breaks. El segundo solo tiene efecto al informar equipos europeos y jornadas críticas; penaliza partidos de visita en esas jornadas.

## Restricciones

- Un partido por equipo y jornada: $\sum_{j\ne i}(x_{ijt}+x_{jit})=1$ para todo $i,t$.
- Doble Round-Robin: $\sum_{t\in J}x_{ijt}=1$ para cada par ordenado $i\ne j$, de modo que cada pareja juega dos veces con localías invertidas.
- Definición y balance de localía: $y_{it}=\sum_{j\ne i}x_{ijt}$ y $\sum_{t\in J}y_{it}=|J|/2$.
- Identificación exacta de breaks: las cotas inferior y superior de $b^H,b^A$ equivalen a productos AND de las localías o visitas consecutivas.
- Como máximo dos localías consecutivas: se prohíben tres jornadas seguidas de local. No existe una restricción equivalente para tres visitas seguidas; estas se penalizan como breaks.
- Restricciones configurables: máximo de clásicos por jornada y no simultaneidad de localía para clubes que comparten estadio.
- Viajes críticos aproximados: $v_{it}=1-y_{it}$ para equipos y jornadas seleccionados.
- $x,y$ son binarias; breaks y $v$ son continuas entre 0 y 1, pero quedan forzadas a valores 0/1 por sus restricciones.

## Qué hace cada celda

1. La primera celda Python localiza el proyecto, selecciona la temporada y declara parámetros vacíos para las reglas operativas opcionales.
2. La segunda resuelve el mismo caso con parámetros predeterminados de Gurobi y con cortes/heurísticas desactivados, con un límite de 3600 segundos por ejecución.
3. La tercera compara estado, incumbente, cota, gap, tiempo y nodos, y muestra el calendario y una muestra de los datos históricos.

En `modelo_calendario.py`, `resolver_instancia` valida y normaliza la entrada, crea las variables $x,y,b^H,b^A,v$, agrega restricciones deportivas y operativas, fija la función objetivo y optimiza. Devuelve el estado del solver, métricas, breaks, viajes aproximados y calendario factible si existe. Sin licencia, la carga de datos puede funcionar, pero la creación/optimización del modelo no.

## Activar Gurobi

Antes de ejecutar las celdas que llaman al solver, abre PowerShell o la terminal de Windows y ejecuta el comando que entrega Gurobi: `grbgetkey CLAVE_ENTREGADA_POR_GUROBI`, reemplazando el texto por la clave real. No guardes la clave en el notebook. Reinicia el kernel después de activar la licencia. Si el comando no se reconoce, instala Gurobi Optimizer para Windows o agrega su carpeta `bin` al `PATH`.

In [ ]:
# Localizar e importar el modelo general compartido.
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")
sys.path.insert(0, str(RUTA_PROYECTO))

from modelo_calendario import resolver_instancia

URLS_TEMPORADA = {
    "2021-2022": "https://www.football-data.co.uk/mmz4281/2122/E0.csv",
    "2022-2023": "https://www.football-data.co.uk/mmz4281/2223/E0.csv",
    "2023-2024": "https://www.football-data.co.uk/mmz4281/2324/E0.csv",
    "2024-2025": "https://www.football-data.co.uk/mmz4281/2425/E0.csv",
    "2025-2026": "https://www.football-data.co.uk/mmz4281/2526/E0.csv",
    "2026-2027": "https://www.football-data.co.uk/mmz4281/2627/E0.csv",
}
TEMPORADA = "2024-2025"
URL_CSV = URLS_TEMPORADA[TEMPORADA]
print(f"Instancia seleccionada: {TEMPORADA}")

Instancia seleccionada: 2024-2025


In [ ]:
# Configurar restricciones operativas y pesos de la función objetivo.
PARAMETROS_MODELO = {
    "clasicos": [],  # Ejemplo: [("Arsenal", "Tottenham")]
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],  # Números entre 1 y 38 para una liga de 20 equipos
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "tiempo_limite": 3600,
    "numero_equipos_esperado": 20,
}

In [ ]:
# Resolver con parámetros por defecto y con cortes/heurísticas desactivados.
configuraciones = {
    "Normal (Gurobi por defecto)": False,
    "Sin cortes ni heurísticas": True,
}
resultados = {}
for nombre, desactivar_cortes_heuristicas in configuraciones.items():
    resultados[nombre] = resolver_instancia(
        TEMPORADA,
        URL_CSV,
        desactivar_cortes_heuristicas=desactivar_cortes_heuristicas,
        **PARAMETROS_MODELO,
    )

# Reporte solicitado para ambas ejecuciones.
reporte = pd.DataFrame(
    [
        {
            "Configuración": nombre,
            "Estado Gurobi": resultado["estado"],
            "Objetivo (ObjVal)": resultado["objetivo"],
            "Cota (ObjBoundC)": resultado["cota"],
            "Gap (MIPGap)": resultado["gap"],
            "Tiempo (s)": resultado["tiempo"],
            "Nodos (NodeCount)": resultado["nodos"],
        }
        for nombre, resultado in resultados.items()
    ]
)
display(reporte)

# Mostrar también el calendario de la primera configuración si se encontró solución.
resultado_normal = resultados["Normal (Gurobi por defecto)"]
print(f"Partidos leídos desde internet: {len(resultado_normal['partidos_historicos'])}")
print(f"Breaks: {resultado_normal['breaks']}")
print(f"Viajes críticos: {resultado_normal['viajes_criticos']}")
display(resultado_normal["calendario"])
display(resultado_normal["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))